# 🏥Healthcare Claims Analysis Project

### 👤 Author: Abhishek Potnis

## 🧠1. Business Problem

### In the world of Healthcare Claims, the costs for treating certain chronic conditions can vary based on different vendors. All of the data used here is taken from the CMS Medicare website: https://www.cms.gov/. Medicare is the government healthcare program for US citizens aged 65+.



###### Member Benefit Data: https://www.cms.gov/Research-Statistics-Data-and-Systems/Downloadable-Public-Use-Files/SynPUFs/Downloads/DE1_0_2009_Beneficiary_Summary_File_Sample_20.zip

###### Outpatient Claim Data: https://www.cms.gov/Research-Statistics-Data-and-Systems/Downloadable-Public-Use-Files/SynPUFs/Downloads/DE1_0_2008_to_2010_Outpatient_Claims_Sample_20.zip

###### User Documentation : https://www.cms.gov/Research-Statistics-Data-and-Systems/Downloadable-Public-Use-Files/SynPUFs/Downloads/SynPUF_DUG.pdf

###### Additional Info: https://www.cms.gov/Research-Statistics-Data-and-Systems/Downloadable-Public-Use-Files/SynPUFs

###### More data about Tables can be seen in this pdf: https://www.cms.gov/files/document/de-10-codebook.pdf-0

#### This project aims to:
- Identify the most common chronic illness combo.
- Identify the chronic illness combo with the highest cost.
- Identify the chronic illness combo with the highest cost per member.
- Identify the most expensive Providers/Vendors.

## 📦2.Data Loading

In [81]:
import pandas as pd

icd_cols = [
    "ICD9_DGNS_CD_10",
    "ICD9_PRCDR_CD_2",
    "ICD9_PRCDR_CD_3",
    "ICD9_PRCDR_CD_4",
    "ICD9_PRCDR_CD_5",
    "ICD9_PRCDR_CD_6",
]
ben_summ = pd.read_csv("/content/DE1_0_2009_Beneficiary_Summary_File_Sample_20.csv")
out_claim = pd.read_csv("/content/DE1_0_2008_to_2010_Outpatient_Claims_Sample_20.csv", dtype={c: "string" for c in icd_cols}, low_memory=False)

## 🤯3. Initial Data Exploration

##### Initial EDA for Benefits Summary Dataset

In [82]:
ben_summ.head()

,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,...,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
0,000002F7E0A96C32,19190701,NaN,2,2,0,5,400,0,0,...,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,00001C24EE7B06AC,19360501,NaN,1,1,0,11,530,12,12,...,2,0.0,0.0,0.0,200.0,40.0,0.0,800.0,260.0,0.0
2,000072CF62193213,19310401,NaN,2,1,0,34,120,12,12,...,2,0.0,0.0,0.0,130.0,70.0,0.0,440.0,30.0,50.0
3,0000DCD33779ED8A,19420501,NaN,2,2,0,11,190,12,12,...,2,0.0,0.0,0.0,90.0,20.0,0.0,930.0,200.0,0.0
4,0000F1EB530967F3,19350401,NaN,2,1,0,23,720,12,12,...,2,0.0,0.0,0.0,70.0,200.0,0.0,4950.0,1340.0,0.0


In [83]:
ben_summ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114641 entries, 0 to 114640
Data columns (total 32 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   DESYNPUF_ID               114641 non-null  object 
 1   BENE_BIRTH_DT             114641 non-null  int64  
 2   BENE_DEATH_DT             1830 non-null    float64
 3   BENE_SEX_IDENT_CD         114641 non-null  int64  
 4   BENE_RACE_CD              114641 non-null  int64  
 5   BENE_ESRD_IND             114641 non-null  object 
 6   SP_STATE_CODE             114641 non-null  int64  
 7   BENE_COUNTY_CD            114641 non-null  int64  
 8   BENE_HI_CVRAGE_TOT_MONS   114641 non-null  int64  
 9   BENE_SMI_CVRAGE_TOT_MONS  114641 non-null  int64  
 10  BENE_HMO_CVRAGE_TOT_MONS  114641 non-null  int64  
 11  PLAN_CVRG_MOS_NUM         114641 non-null  int64  
 12  SP_ALZHDMTA               114641 non-null  int64  
 13  SP_CHF                    114641 non-null  i

In [84]:
ben_summ.describe()

,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,BENE_HMO_CVRAGE_TOT_MONS,PLAN_CVRG_MOS_NUM,...,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
count,1.146410e+05,1.830000e+03,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,...,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000,114641.000000
mean,1.936404e+07,2.009063e+07,1.554296,1.283747,25.734946,366.369885,11.228653,11.113284,3.332289,9.070237,...,1.947767,2158.440000,247.422057,101.529994,762.027111,234.910634,30.711177,1337.588036,374.557532,21.111557
std,1.256401e+05,3.441484e+02,0.497045,0.751824,15.571550,266.031040,2.926545,3.093597,5.303927,4.869408,...,0.222497,7167.900814,781.243963,1951.358679,1876.649919,539.392067,414.684301,1525.171106,423.541047,96.009641
min,1.909010e+07,2.009010e+07,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,-8000.000000,0.000000,0.000000,-90.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.928110e+07,2.009030e+07,1.000000,1.000000,11.000000,141.000000,12.000000,12.000000,0.000000,6.000000,...,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,90.000000,20.000000,0.000000
50%,1.936050e+07,2.009060e+07,2.000000,1.000000,25.000000,330.000000,12.000000,12.000000,0.000000,12.000000,...,2.000000,0.000000,0.000000,0.000000,120.000000,30.000000,0.000000,900.000000,250.000000,0.000000
75%,1.942030e+07,2.009090e+07,2.000000,1.000000,39.000000,550.000000,12.000000,12.000000,12.000000,12.000000,...,2.000000,0.000000,0.000000,0.000000,800.000000,240.000000,0.000000,1980.000000,560.000000,0.000000
max,1.983120e+07,2.009120e+07,2.000000,5.000000,54.000000,999.000000,12.000000,12.000000,12.000000,12.000000,...,2.000000,136000.000000,39340.000000,86000.000000,47010.000000,11480.000000,21000.000000,16930.000000,4410.000000,2400.000000


In [85]:
ben_summ.dtypes

,0
DESYNPUF_ID,object
BENE_BIRTH_DT,int64
BENE_DEATH_DT,float64
BENE_SEX_IDENT_CD,int64
BENE_RACE_CD,int64
BENE_ESRD_IND,object
SP_STATE_CODE,int64
BENE_COUNTY_CD,int64
BENE_HI_CVRAGE_TOT_MONS,int64
BENE_SMI_CVRAGE_TOT_MONS,int64


In [86]:
ben_summ.columns

Index(['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD',
       'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD',
       'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS',
       'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA',
       'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN',
       'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA',
       'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP',
       'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR'],
      dtype='object')

In [87]:
#Examing Null Heavy Columns
ben_summ.isna().sum().sort_values(ascending=False)

,0
BENE_DEATH_DT,112811
DESYNPUF_ID,0
BENE_BIRTH_DT,0
BENE_SEX_IDENT_CD,0
BENE_RACE_CD,0
BENE_ESRD_IND,0
SP_STATE_CODE,0
BENE_COUNTY_CD,0
BENE_HI_CVRAGE_TOT_MONS,0
BENE_SMI_CVRAGE_TOT_MONS,0


##### Initial EDA for Outpatient Claims Dataset

In [88]:
out_claim.head()

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00001C24EE7B06AC,684562269783396,1,20090404.0,20090404.0,1100SK,200.0,0.0,1.298827e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00001C24EE7B06AC,684012269893042,1,20100310.0,20100310.0,1100SK,500.0,0.0,1.298827e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000072CF62193213,684012269540703,1,20080130.0,20080130.0,1000AH,50.0,0.0,8.929521e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000072CF62193213,684472269696971,1,20080301.0,20080301.0,1000AH,70.0,0.0,8.382688e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,000072CF62193213,684122269778934,1,20080322.0,20080322.0,3400HK,40.0,0.0,4.404237e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [89]:
out_claim.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790044 entries, 0 to 790043
Data columns (total 76 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   DESYNPUF_ID                     790044 non-null  object 
 1   CLM_ID                          790044 non-null  int64  
 2   SEGMENT                         790044 non-null  int64  
 3   CLM_FROM_DT                     779016 non-null  float64
 4   CLM_THRU_DT                     779016 non-null  float64
 5   PRVDR_NUM                       790044 non-null  object 
 6   CLM_PMT_AMT                     790044 non-null  float64
 7   NCH_PRMRY_PYR_CLM_PD_AMT        790044 non-null  float64
 8   AT_PHYSN_NPI                    772604 non-null  float64
 9   OP_PHYSN_NPI                    133737 non-null  float64
 10  OT_PHYSN_NPI                    255940 non-null  float64
 11  NCH_BENE_BLOOD_DDCTBL_LBLTY_AM  790044 non-null  float64
 12  ICD9_DGNS_CD_1  

In [90]:
out_claim.describe()

,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,OT_PHYSN_NPI,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,ICD9_PRCDR_CD_1,NCH_BENE_PTB_DDCTBL_AMT,NCH_BENE_PTB_COINSRNC_AMT,HCPCS_CD_45
count,7.900440e+05,790044.000000,7.790160e+05,7.790160e+05,790044.000000,790044.000000,7.726040e+05,1.337370e+05,2.559400e+05,790044.000000,205.000000,790044.000000,790044.000000,0.0
mean,6.845019e+14,1.013959,2.008923e+07,2.008927e+07,282.599096,10.425976,4.973535e+09,4.939105e+09,4.902958e+09,0.008227,6253.082927,2.812843,83.796118,NaN
std,2.858064e+11,0.117320,7.473580e+03,7.470445e+03,569.602877,235.285339,2.877391e+09,2.885570e+09,2.891626e+09,2.028209,3000.040882,15.512668,178.725958,NaN
min,6.840123e+14,1.000000,2.007121e+07,2.008010e+07,-100.000000,0.000000,1.024080e+05,1.005544e+06,1.024080e+05,0.000000,54.000000,0.000000,0.000000,NaN
25%,6.842523e+14,1.000000,2.008092e+07,2.008092e+07,40.000000,0.000000,2.520575e+09,2.469459e+09,2.437455e+09,0.000000,3893.000000,0.000000,0.000000,NaN
50%,6.845023e+14,1.000000,2.009042e+07,2.009043e+07,80.000000,0.000000,4.905258e+09,4.866312e+09,4.781420e+09,0.000000,5551.000000,0.000000,20.000000,NaN
75%,6.847523e+14,1.000000,2.009120e+07,2.009120e+07,200.000000,0.000000,7.505951e+09,7.470920e+09,7.494005e+09,0.000000,8962.000000,0.000000,70.000000,NaN
max,6.849923e+14,2.000000,2.010123e+07,2.010123e+07,3300.000000,14000.000000,9.999658e+09,9.999615e+09,9.999886e+09,900.000000,9955.000000,200.000000,1100.000000,NaN


In [91]:
out_claim.dtypes

,0
DESYNPUF_ID,object
CLM_ID,int64
SEGMENT,int64
CLM_FROM_DT,float64
CLM_THRU_DT,float64
...,...
HCPCS_CD_41,object
HCPCS_CD_42,object
HCPCS_CD_43,object
HCPCS_CD_44,object


In [92]:
out_claim.columns

Index(['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT',
       'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI',
       'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
       'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4',
       'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8',
       'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10', 'ICD9_PRCDR_CD_1',
       'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4',
       'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6', 'NCH_BENE_PTB_DDCTBL_AMT',
       'NCH_BENE_PTB_COINSRNC_AMT', 'ADMTNG_ICD9_DGNS_CD', 'HCPCS_CD_1',
       'HCPCS_CD_2', 'HCPCS_CD_3', 'HCPCS_CD_4', 'HCPCS_CD_5', 'HCPCS_CD_6',
       'HCPCS_CD_7', 'HCPCS_CD_8', 'HCPCS_CD_9', 'HCPCS_CD_10', 'HCPCS_CD_11',
       'HCPCS_CD_12', 'HCPCS_CD_13', 'HCPCS_CD_14', 'HCPCS_CD_15',
       'HCPCS_CD_16', 'HCPCS_CD_17', 'HCPCS_CD_18', 'HCPCS_CD_19',
       'HCPCS_CD_20', 'HCPCS_CD_21', 'HCPCS_CD_22', 'HC

In [93]:
out_claim.isna().sum().sort_values(ascending=False).head(70)

,0
HCPCS_CD_45,790044
ICD9_PRCDR_CD_6,790030
ICD9_PRCDR_CD_5,790026
ICD9_PRCDR_CD_4,790018
ICD9_PRCDR_CD_3,789991
...,...
CLM_THRU_DT,11028
ICD9_DGNS_CD_1,5514
SEGMENT,0
NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,0


## 🧹4.Data Cleaning and Transformation

##### Data Cleaning and Transformation for the Benefits Summary Dataset

In [94]:
#Dropping the columns that are not needed for the analysis
ben_summ_cols_to_drop = ["BENE_HI_CVRAGE_TOT_MONS", "BENE_SMI_CVRAGE_TOT_MONS","BENE_HMO_CVRAGE_TOT_MONS","PLAN_CVRG_MOS_NUM","MEDREIMB_IP","BENRES_IP","PPPYMT_IP","MEDREIMB_CAR","BENRES_CAR","PPPYMT_CAR"]
ben_summ = ben_summ.drop(columns=ben_summ_cols_to_drop)

In [95]:
#Since Beneficiary Death Date is a null heavy column also dropping that.
ben_summ = ben_summ.drop(columns="BENE_DEATH_DT")

In [96]:
ben_summ.columns

Index(['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD',
       'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'SP_ALZHDMTA',
       'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN',
       'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA',
       'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP'],
      dtype='object')

In [97]:
#Renaming the columns to be more descriptive
ben_summ_cols_to_rename = {"DESYNPUF_ID": "beneficiary_id",
                        "BENE_BIRTH_DT": "birth_date",
                        "BENE_DEATH_DT": "death_date",
                        "BENE_SEX_IDENT_CD": "sex",
                        "BENE_RACE_CD": "race",
                        "BENE_ESRD_IND" : "renal_disease_indicator",
                        "SP_STATE_CODE" : "state_code",
                        "BENE_COUNTY_CD" : "county",
                        "SP_ALZHDMTA" : "alzheimer",
                        "SP_CHF" : "heart_failure",
                        "SP_CHRNKIDN": "chronic_kidney_disease",
                        "SP_CNCR" : "cancer",
                        "SP_COPD" : "obstructive_pulmonary_disease",
                        "SP_DEPRESSN" : "depression",
                        "SP_DIABETES" : "diabetes",
                        "SP_ISCHMCHT" : "ischemic_heart_disease",
                        "SP_OSTEOPRS" : "osteoporosis",
                        "SP_RA_OA" : "arthritis",
                        "SP_STRKETIA" : "stroke",
                        "MEDREIMB_OP" : "medicare_reimbursement_amt",
                        "BENRES_OP" : "beneficiary_responsibility_amt",
                        "PPPYMT_OP" : "primary_payer_reimbursement_amt"}


In [98]:
ben_summ = ben_summ.rename(columns=ben_summ_cols_to_rename)
ben_summ.columns

Index(['beneficiary_id', 'birth_date', 'sex', 'race',
       'renal_disease_indicator', 'state_code', 'county', 'alzheimer',
       'heart_failure', 'chronic_kidney_disease', 'cancer',
       'obstructive_pulmonary_disease', 'depression', 'diabetes',
       'ischemic_heart_disease', 'osteoporosis', 'arthritis', 'stroke',
       'medicare_reimbursement_amt', 'beneficiary_responsibility_amt',
       'primary_payer_reimbursement_amt'],
      dtype='object')

In [99]:
#Transforming the data types of the columns
ben_summ.dtypes

,0
beneficiary_id,object
birth_date,int64
sex,int64
race,int64
renal_disease_indicator,object
state_code,int64
county,int64
alzheimer,int64
heart_failure,int64
chronic_kidney_disease,int64


In [100]:
#Transforming all Date columns to datetime
ben_summ["birth_date"] = pd.to_datetime(ben_summ["birth_date"],format="%Y%m%d")
ben_summ["birth_date"]

,birth_date
0,1919-07-01
1,1936-05-01
2,1931-04-01
3,1942-05-01
4,1935-04-01
...,...
114636,1932-05-01
114637,1933-06-01
114638,1940-11-01
114639,1926-11-01


In [101]:
ben_summ.dtypes

,0
beneficiary_id,object
birth_date,datetime64[ns]
sex,int64
race,int64
renal_disease_indicator,object
state_code,int64
county,int64
alzheimer,int64
heart_failure,int64
chronic_kidney_disease,int64


In [102]:
#Transforming the Beneficiary_Sex column
ben_summ["sex"].dtype
ben_summ["sex"].value_counts()
ben_summ["sex"] = ben_summ["sex"].map({1: "Male", 2: "Female"})
ben_summ["sex"].value_counts()

,count
sex,
Female,63545
Male,51096


In [103]:
#Transforming the Beneficiary_Race column
ben_summ["race"].dtype
ben_summ["race"].value_counts()
ben_summ["race"] = ben_summ["race"].map({1: "White", 2: "Black", 3: "Others", 5: "Hispanic",})
ben_summ["race"].value_counts()



,count
race,
White,94945
Black,12141
Others,4916
Hispanic,2639


In [104]:
ben_summ["renal_disease_indicator"]

,renal_disease_indicator
0,0
1,0
2,0
3,0
4,0
...,...
114636,0
114637,0
114638,0
114639,0


In [105]:
#Transforming the renal_disease_indicator column
ben_summ["renal_disease_indicator"].value_counts()
ben_summ["renal_disease_indicator"] = ben_summ["renal_disease_indicator"].map({"Y": "Yes", "0": "No"})
ben_summ["renal_disease_indicator"].value_counts()


,count
renal_disease_indicator,
No,103781
Yes,10860


In [106]:
ben_summ.columns

Index(['beneficiary_id', 'birth_date', 'sex', 'race',
       'renal_disease_indicator', 'state_code', 'county', 'alzheimer',
       'heart_failure', 'chronic_kidney_disease', 'cancer',
       'obstructive_pulmonary_disease', 'depression', 'diabetes',
       'ischemic_heart_disease', 'osteoporosis', 'arthritis', 'stroke',
       'medicare_reimbursement_amt', 'beneficiary_responsibility_amt',
       'primary_payer_reimbursement_amt'],
      dtype='object')

In [107]:
ben_summ[["state_code", "county"]].isna().sum()

#No transformation needed for the state_code and county columns

,0
state_code,0
county,0


In [108]:
#Transforming all of the chronic conditions columns to boolean
ben_summ["alzheimer"] = ben_summ["alzheimer"].map({1: True, 2: False})
ben_summ["alzheimer"].value_counts()
ben_summ["heart_failure"] = ben_summ["heart_failure"].map({1: True, 2: False})
ben_summ["heart_failure"].value_counts()
ben_summ["chronic_kidney_disease"] = ben_summ["chronic_kidney_disease"].map({1: True, 2: False})
ben_summ["chronic_kidney_disease"].value_counts()
ben_summ["cancer"] = ben_summ["cancer"].map({1: True, 2: False})
ben_summ["cancer"].value_counts()
ben_summ["obstructive_pulmonary_disease"] = ben_summ["obstructive_pulmonary_disease"].map({1: True, 2: False})
ben_summ["obstructive_pulmonary_disease"].value_counts()
ben_summ["depression"] = ben_summ["depression"].map({1: True, 2: False})
ben_summ["depression"].value_counts()
ben_summ["diabetes"] = ben_summ["diabetes"].map({1: True, 2: False})
ben_summ["diabetes"].value_counts()
ben_summ["ischemic_heart_disease"] = ben_summ["ischemic_heart_disease"].map({1: True, 2: False})
ben_summ["ischemic_heart_disease"].value_counts()
ben_summ["osteoporosis"] = ben_summ["osteoporosis"].map({1: True, 2: False})
ben_summ["osteoporosis"].value_counts()
ben_summ["arthritis"] = ben_summ["arthritis"].map({1: True, 2: False})
ben_summ["arthritis"].value_counts()
ben_summ["stroke"] = ben_summ["stroke"].map({1: True, 2: False})
ben_summ["stroke"].value_counts()

,count
stroke,
False,108653
True,5988


In [109]:
ben_summ.value_counts()

,,,,,,,,,,,,,,,,,,,,,count
beneficiary_id,birth_date,sex,race,renal_disease_indicator,state_code,county,alzheimer,heart_failure,chronic_kidney_disease,cancer,obstructive_pulmonary_disease,depression,diabetes,ischemic_heart_disease,osteoporosis,arthritis,stroke,medicare_reimbursement_amt,beneficiary_responsibility_amt,primary_payer_reimbursement_amt,
FFFE6F53545F974E,1921-08-01,Female,White,No,36,10,True,False,True,False,False,False,True,True,False,False,False,600.0,50.0,0.0,1
FFF762DC60327F8D,1922-01-01,Male,Hispanic,No,5,530,True,True,True,False,False,True,True,True,False,False,False,230.0,100.0,0.0,1
FFF6E216FE04BF21,1921-08-01,Female,White,No,33,380,False,False,False,False,False,False,False,True,False,False,False,80.0,40.0,0.0,1
FFF642EE2B954003,1933-03-01,Male,White,No,44,780,False,False,False,False,False,False,False,False,False,False,False,0.0,0.0,0.0,1
FFF62AB8537403E2,1935-05-01,Male,Hispanic,No,31,350,False,False,False,False,False,False,False,False,False,False,False,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0001B495F55E2DFA,1941-08-01,Female,White,No,34,170,False,False,False,False,False,True,True,False,True,True,False,1100.0,400.0,0.0,1
0000F1EB530967F3,1935-04-01,Female,White,No,23,720,True,True,False,True,False,True,True,True,True,True,False,70.0,200.0,0.0,1
0000DCD33779ED8A,1942-05-01,Female,Black,No,11,190,True,True,False,True,False,False,False,False,False,False,False,90.0,20.0,0.0,1


In [110]:
ben_summ["primary_payer_reimbursement_amt"].value_counts()
ben_summ["medicare_reimbursement_amt"].value_counts()
ben_summ["beneficiary_responsibility_amt"].value_counts()
# No transformation needed for the reimbursement columns.

,count
beneficiary_responsibility_amt,
0.0,48718
20.0,4078
10.0,3591
100.0,2720
30.0,2658
...,...
9300.0,1
5870.0,1
5520.0,1


In [111]:
#Convert all Chronic Conditions columns to a single categorical variable concatenating multiple true diagnoses.
ben_summ.columns
chronic_conditions_cols = ["alzheimer", "heart_failure", "chronic_kidney_disease", "cancer", "obstructive_pulmonary_disease", "depression", "diabetes", "ischemic_heart_disease", "osteoporosis", "arthritis", "stroke"]
ben_summ["chronic_conditions"] = ben_summ[chronic_conditions_cols].apply(
    lambda x: " & ".join(x[x == True].index.tolist()) or "None", axis=1
)
ben_summ["chronic_conditions"].value_counts()
ben_summ["chronic_conditions"].head(20)



,chronic_conditions
0,None
1,ischemic_heart_disease
2,None
3,alzheimer & heart_failure & cancer
4,alzheimer & heart_failure & cancer & depressio...
5,depression & diabetes & osteoporosis & arthritis
6,depression & diabetes & ischemic_heart_disease
7,heart_failure & depression & diabetes
8,None
9,alzheimer & cancer & obstructive_pulmonary_dis...


In [112]:
#If a member has 3 or more chronic conditions, categorise these as “Multiple”

ben_summ["chronic_conditions"] = ben_summ["chronic_conditions"].apply(lambda x: "Multiple" if len(x.split(" & ")) >= 3 else x)
ben_summ["chronic_conditions"].value_counts()
ben_summ["chronic_conditions"].head(20)

,chronic_conditions
0,None
1,ischemic_heart_disease
2,None
3,Multiple
4,Multiple
5,Multiple
6,Multiple
7,Multiple
8,None
9,Multiple


##### Data Cleaning and Transformation for the Outpatient Claims Dataset

In [113]:
out_claim.columns

Index(['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT',
       'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI',
       'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
       'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4',
       'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8',
       'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10', 'ICD9_PRCDR_CD_1',
       'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4',
       'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6', 'NCH_BENE_PTB_DDCTBL_AMT',
       'NCH_BENE_PTB_COINSRNC_AMT', 'ADMTNG_ICD9_DGNS_CD', 'HCPCS_CD_1',
       'HCPCS_CD_2', 'HCPCS_CD_3', 'HCPCS_CD_4', 'HCPCS_CD_5', 'HCPCS_CD_6',
       'HCPCS_CD_7', 'HCPCS_CD_8', 'HCPCS_CD_9', 'HCPCS_CD_10', 'HCPCS_CD_11',
       'HCPCS_CD_12', 'HCPCS_CD_13', 'HCPCS_CD_14', 'HCPCS_CD_15',
       'HCPCS_CD_16', 'HCPCS_CD_17', 'HCPCS_CD_18', 'HCPCS_CD_19',
       'HCPCS_CD_20', 'HCPCS_CD_21', 'HCPCS_CD_22', 'HC

In [114]:
#Dropping the ICD9 columns as most of them are NaNs and not needed for the analysis.
out_claim.isna().sum().sort_values(ascending=False).head(68)
out_claim.columns[12:28]
out_claim = out_claim.drop(columns=out_claim.columns[12:28])
out_claim.columns









Index(['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT',
       'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI',
       'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
       'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT',
       'ADMTNG_ICD9_DGNS_CD', 'HCPCS_CD_1', 'HCPCS_CD_2', 'HCPCS_CD_3',
       'HCPCS_CD_4', 'HCPCS_CD_5', 'HCPCS_CD_6', 'HCPCS_CD_7', 'HCPCS_CD_8',
       'HCPCS_CD_9', 'HCPCS_CD_10', 'HCPCS_CD_11', 'HCPCS_CD_12',
       'HCPCS_CD_13', 'HCPCS_CD_14', 'HCPCS_CD_15', 'HCPCS_CD_16',
       'HCPCS_CD_17', 'HCPCS_CD_18', 'HCPCS_CD_19', 'HCPCS_CD_20',
       'HCPCS_CD_21', 'HCPCS_CD_22', 'HCPCS_CD_23', 'HCPCS_CD_24',
       'HCPCS_CD_25', 'HCPCS_CD_26', 'HCPCS_CD_27', 'HCPCS_CD_28',
       'HCPCS_CD_29', 'HCPCS_CD_30', 'HCPCS_CD_31', 'HCPCS_CD_32',
       'HCPCS_CD_33', 'HCPCS_CD_34', 'HCPCS_CD_35', 'HCPCS_CD_36',
       'HCPCS_CD_37', 'HCPCS_CD_38', 'HCPCS_CD_39', 'HCPCS_CD_40',
       'HCPCS_CD_41', 'HCPCS

In [115]:
#Dropping the HCPCS columns as most of them are NaNs and not needed for the analysis.
out_claim.isna().sum().sort_values(ascending=False).head(68)
out_claim.columns[15:]
out_claim = out_claim.drop(columns=out_claim.columns[15:])
out_claim.columns

Index(['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT',
       'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI',
       'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
       'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT',
       'ADMTNG_ICD9_DGNS_CD'],
      dtype='object')

In [116]:
#Dropping the ADMTNG_ICD9_DGNS_CD column as it is not needed for the analysis.
out_claim.isna().sum().sort_values(ascending=False)
out_claim = out_claim.drop(columns=["ADMTNG_ICD9_DGNS_CD"])
out_claim.columns


Index(['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT',
       'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI',
       'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
       'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT'],
      dtype='object')

In [117]:
out_claim.isna().sum().sort_values(ascending=False)
#Dropping all physician columns, since these are Null Heavy
out_claim = out_claim.drop(columns=["AT_PHYSN_NPI","OP_PHYSN_NPI","OT_PHYSN_NPI"])

In [118]:
#Renaming the columns to be more descriptive
out_claim.columns
out_claim_cols_to_rename = {"DESYNPUF_ID": "beneficiary_id",
"CLM_ID": "claim_id",
"SEGMENT": "segment",
"CLM_FROM_DT": "claim_from_date",
"CLM_THRU_DT": "claim_thru_date",
"PRVDR_NUM": "provider_number",
"CLM_PMT_AMT": "claim_payment_amt",
"NCH_PRMRY_PYR_CLM_PD_AMT": "primary_payer_claim_paid_amt",
"AT_PHYSN_NPI": "attending_physician",
"OP_PHYSN_NPI": "operating_physician",
"OT_PHYSN_NPI": "other_physician",
"NCH_BENE_BLOOD_DDCTBL_LBLTY_AM": "blood_deductible_liability_amount",
"NCH_BENE_PTB_DDCTBL_AMT": "part_b_deductible_amount",
"NCH_BENE_PTB_COINSRNC_AMT": "part_b_coinsurance_amount"}
out_claim = out_claim.rename(columns=out_claim_cols_to_rename)
out_claim.columns

Index(['beneficiary_id', 'claim_id', 'segment', 'claim_from_date',
       'claim_thru_date', 'provider_number', 'claim_payment_amt',
       'primary_payer_claim_paid_amt', 'blood_deductible_liability_amount',
       'part_b_deductible_amount', 'part_b_coinsurance_amount'],
      dtype='object')

In [119]:
out_claim.dtypes
out_claim.head(20)

,beneficiary_id,claim_id,segment,claim_from_date,claim_thru_date,provider_number,claim_payment_amt,primary_payer_claim_paid_amt,blood_deductible_liability_amount,part_b_deductible_amount,part_b_coinsurance_amount
0,00001C24EE7B06AC,684562269783396,1,20090404.0,20090404.0,1100SK,200.0,0.0,0.0,0.0,40.0
1,00001C24EE7B06AC,684012269893042,1,20100310.0,20100310.0,1100SK,500.0,0.0,0.0,0.0,0.0
2,000072CF62193213,684012269540703,1,20080130.0,20080130.0,1000AH,50.0,0.0,0.0,0.0,0.0
3,000072CF62193213,684472269696971,1,20080301.0,20080301.0,1000AH,70.0,0.0,0.0,0.0,20.0
4,000072CF62193213,684122269778934,1,20080322.0,20080322.0,3400HK,40.0,0.0,0.0,0.0,10.0
5,000072CF62193213,684652269610164,1,20080512.0,20080512.0,3400KP,10.0,0.0,0.0,0.0,0.0
6,000072CF62193213,684382269804352,1,20080615.0,20080615.0,3400WD,500.0,0.0,0.0,0.0,100.0
7,000072CF62193213,684252269507042,1,20080812.0,20080812.0,3400ZQ,30.0,0.0,0.0,0.0,0.0
8,000072CF62193213,684342269576155,1,20080810.0,20080830.0,3400HK,100.0,0.0,0.0,0.0,10.0
9,000072CF62193213,684372269909907,1,20081110.0,20081110.0,3465CJ,50.0,0.0,0.0,0.0,10.0


In [120]:
#Transforming the date columns to datetime
out_claim["claim_from_date"] = pd.to_datetime(out_claim["claim_from_date"],format="%Y%m%d")
out_claim["claim_thru_date"] = pd.to_datetime(out_claim["claim_thru_date"],format="%Y%m%d")
out_claim["claim_from_date"]
out_claim["claim_thru_date"]
out_claim.dtypes


,0
beneficiary_id,object
claim_id,int64
segment,int64
claim_from_date,datetime64[ns]
claim_thru_date,datetime64[ns]
provider_number,object
claim_payment_amt,float64
primary_payer_claim_paid_amt,float64
blood_deductible_liability_amount,float64
part_b_deductible_amount,float64


In [121]:
out_claim.head(20)

,beneficiary_id,claim_id,segment,claim_from_date,claim_thru_date,provider_number,claim_payment_amt,primary_payer_claim_paid_amt,blood_deductible_liability_amount,part_b_deductible_amount,part_b_coinsurance_amount
0,00001C24EE7B06AC,684562269783396,1,2009-04-04,2009-04-04,1100SK,200.0,0.0,0.0,0.0,40.0
1,00001C24EE7B06AC,684012269893042,1,2010-03-10,2010-03-10,1100SK,500.0,0.0,0.0,0.0,0.0
2,000072CF62193213,684012269540703,1,2008-01-30,2008-01-30,1000AH,50.0,0.0,0.0,0.0,0.0
3,000072CF62193213,684472269696971,1,2008-03-01,2008-03-01,1000AH,70.0,0.0,0.0,0.0,20.0
4,000072CF62193213,684122269778934,1,2008-03-22,2008-03-22,3400HK,40.0,0.0,0.0,0.0,10.0
5,000072CF62193213,684652269610164,1,2008-05-12,2008-05-12,3400KP,10.0,0.0,0.0,0.0,0.0
6,000072CF62193213,684382269804352,1,2008-06-15,2008-06-15,3400WD,500.0,0.0,0.0,0.0,100.0
7,000072CF62193213,684252269507042,1,2008-08-12,2008-08-12,3400ZQ,30.0,0.0,0.0,0.0,0.0
8,000072CF62193213,684342269576155,1,2008-08-10,2008-08-30,3400HK,100.0,0.0,0.0,0.0,10.0
9,000072CF62193213,684372269909907,1,2008-11-10,2008-11-10,3465CJ,50.0,0.0,0.0,0.0,10.0


In [122]:
ben_summ.head(20)

,beneficiary_id,birth_date,sex,race,renal_disease_indicator,state_code,county,alzheimer,heart_failure,chronic_kidney_disease,...,depression,diabetes,ischemic_heart_disease,osteoporosis,arthritis,stroke,medicare_reimbursement_amt,beneficiary_responsibility_amt,primary_payer_reimbursement_amt,chronic_conditions
0,000002F7E0A96C32,1919-07-01,Female,Black,No,5,400,False,False,False,...,False,False,False,False,False,False,0.0,0.0,0.0,None
1,00001C24EE7B06AC,1936-05-01,Male,White,No,11,530,False,False,False,...,False,False,True,False,False,False,200.0,40.0,0.0,ischemic_heart_disease
2,000072CF62193213,1931-04-01,Female,White,No,34,120,False,False,False,...,False,False,False,False,False,False,130.0,70.0,0.0,None
3,0000DCD33779ED8A,1942-05-01,Female,Black,No,11,190,True,True,False,...,False,False,False,False,False,False,90.0,20.0,0.0,Multiple
4,0000F1EB530967F3,1935-04-01,Female,White,No,23,720,True,True,False,...,True,True,True,True,True,False,70.0,200.0,0.0,Multiple
5,0001B495F55E2DFA,1941-08-01,Female,White,No,34,170,False,False,False,...,True,True,False,True,True,False,1100.0,400.0,0.0,Multiple
6,00028CFDA8612B87,1943-10-01,Female,White,No,14,740,False,False,False,...,True,True,True,False,False,False,1300.0,110.0,0.0,Multiple
7,00036DA073115F08,1943-09-01,Female,White,No,36,310,False,True,False,...,True,True,False,False,False,False,2280.0,280.0,0.0,Multiple
8,0003A64D9776B051,1942-09-01,Female,White,No,45,100,False,False,False,...,False,False,False,False,False,False,0.0,0.0,0.0,None
9,0003D0FBC87B8600,1943-12-01,Male,Black,No,3,130,True,False,False,...,True,True,True,False,False,False,270.0,110.0,0.0,Multiple


## 🤝5.Merging the Datasets

In [123]:
#Left joining the two datasets on the beneficiary_id column
merged_df = out_claim.merge(ben_summ, on="beneficiary_id", how="left")
merged_df.head(20)

,beneficiary_id,claim_id,segment,claim_from_date,claim_thru_date,provider_number,claim_payment_amt,primary_payer_claim_paid_amt,blood_deductible_liability_amount,part_b_deductible_amount,...,depression,diabetes,ischemic_heart_disease,osteoporosis,arthritis,stroke,medicare_reimbursement_amt,beneficiary_responsibility_amt,primary_payer_reimbursement_amt,chronic_conditions
0,00001C24EE7B06AC,684562269783396,1,2009-04-04,2009-04-04,1100SK,200.0,0.0,0.0,0.0,...,False,False,True,False,False,False,200.0,40.0,0.0,ischemic_heart_disease
1,00001C24EE7B06AC,684012269893042,1,2010-03-10,2010-03-10,1100SK,500.0,0.0,0.0,0.0,...,False,False,True,False,False,False,200.0,40.0,0.0,ischemic_heart_disease
2,000072CF62193213,684012269540703,1,2008-01-30,2008-01-30,1000AH,50.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
3,000072CF62193213,684472269696971,1,2008-03-01,2008-03-01,1000AH,70.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
4,000072CF62193213,684122269778934,1,2008-03-22,2008-03-22,3400HK,40.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
5,000072CF62193213,684652269610164,1,2008-05-12,2008-05-12,3400KP,10.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
6,000072CF62193213,684382269804352,1,2008-06-15,2008-06-15,3400WD,500.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
7,000072CF62193213,684252269507042,1,2008-08-12,2008-08-12,3400ZQ,30.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
8,000072CF62193213,684342269576155,1,2008-08-10,2008-08-30,3400HK,100.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None
9,000072CF62193213,684372269909907,1,2008-11-10,2008-11-10,3465CJ,50.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,None


In [124]:
merged_df.info()
merged_df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790044 entries, 0 to 790043
Data columns (total 32 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   beneficiary_id                     790044 non-null  object        
 1   claim_id                           790044 non-null  int64         
 2   segment                            790044 non-null  int64         
 3   claim_from_date                    779016 non-null  datetime64[ns]
 4   claim_thru_date                    779016 non-null  datetime64[ns]
 5   provider_number                    790044 non-null  object        
 6   claim_payment_amt                  790044 non-null  float64       
 7   primary_payer_claim_paid_amt       790044 non-null  float64       
 8   blood_deductible_liability_amount  790044 non-null  float64       
 9   part_b_deductible_amount           790044 non-null  float64       
 10  part_b_coinsurance_a

,0
beneficiary_id,object
claim_id,int64
segment,int64
claim_from_date,datetime64[ns]
claim_thru_date,datetime64[ns]
provider_number,object
claim_payment_amt,float64
primary_payer_claim_paid_amt,float64
blood_deductible_liability_amount,float64
part_b_deductible_amount,float64


## 📊6. Getting Basic Summaries from the Merged Dataset

#### Displaying Chronic Illness Trends

In [125]:
#Displaying Chronic Illness Trends
print("Most common chronic illness combination:")
merged_df["chronic_conditions"].value_counts()


Most common chronic illness combination:


,count
chronic_conditions,
Multiple,604659
None,35835
ischemic_heart_disease,19349
diabetes & ischemic_heart_disease,14273
diabetes,11196
...,...
osteoporosis & stroke,56
arthritis & stroke,49
chronic_kidney_disease & stroke,36


***********************************************

#### Identifying the Chronic Illness Combination with the Highest Cost

In [126]:
#Identifying the most expensive Chronic Illness Combination
# Total outpatient claim payment (CMS CLM_PMT_AMT) by chronic condition combination
cost_by_chronic = (
    merged_df.groupby("chronic_conditions", dropna=False, observed=False)["claim_payment_amt"]
    .agg(total_claim_cost="sum", claim_rows="size")
    .sort_values("total_claim_cost", ascending=False)
)
print("Top 5 combinations by total claim payment on outpatient claims:")
cost_by_chronic.head(5)

Top 5 combinations by total claim payment on outpatient claims:


,total_claim_cost,claim_rows
chronic_conditions,,
Multiple,182385300.0,604659
None,7755010.0,35835
ischemic_heart_disease,4334760.0,19349
diabetes & ischemic_heart_disease,3100730.0,14273
diabetes,2360100.0,11196


****************

#### Identifying the Chronic Illness Combination with the Highest Cost Per Member

In [127]:
#Identifying the most expensive Chronic Illness Combination per member
cost_by_chronic_member = (
    merged_df.groupby(["chronic_conditions"])["claim_payment_amt"]
    .agg(average_member_cost="mean", claim_rows="size")
    .sort_values("average_member_cost", ascending=False)
)
print("Top 10 combinations by average member cost on outpatient claims:")
cost_by_chronic_member.head(10)


Top 10 combinations by average member cost on outpatient claims:


,average_member_cost,claim_rows
chronic_conditions,,
chronic_kidney_disease & stroke,453.055556,36
chronic_kidney_disease & diabetes,339.135338,1862
chronic_kidney_disease,332.031144,1477
cancer & stroke,315.000000,20
Multiple,301.633317,604659
cancer,289.046275,1772
alzheimer & chronic_kidney_disease,284.185464,399
obstructive_pulmonary_disease & stroke,274.545455,11
osteoporosis & stroke,271.964286,56


******

#### Identifying the Most Expensive Provider/Vendor

In [128]:
chronic_claims = merged_df[merged_df["chronic_conditions"] != "None"]

provider_chronic = (
    chronic_claims.groupby("provider_number", as_index=False)
    .agg(
        total_medicare_paid=("claim_payment_amt", "sum"),
        outpatient_claims=("claim_payment_amt", "size"),
        avg_payment_per_claim=("claim_payment_amt", "mean"),
    )
    .sort_values("total_medicare_paid", ascending=False)
    .reset_index(drop=True)
)

print("Top 10 providers by total Medicare payment on this outpatient file (synthetic sample):")
provider_chronic.head(10)

Top 10 providers by total Medicare payment on this outpatient file (synthetic sample):


,provider_number,total_medicare_paid,outpatient_claims,avg_payment_per_claim
0,0502NA,2689190.0,9427,285.264665
1,0505TK,1622340.0,5201,311.928475
2,2201DT,1507290.0,5305,284.126296
3,2301GU,1320220.0,4195,314.712753
4,1000VH,1180850.0,3535,334.045262
5,4500NJ,1080670.0,3631,297.623244
6,3400ZQ,1048940.0,3746,280.016017
7,3200GV,949670.0,2411,393.890502
8,1700JJ,908760.0,3172,286.494325
9,1402JQ,895720.0,2841,315.283351


***************************

## 📂6.Transforming the Merged Dataset to CSV format for further Data Visualization

In [129]:
merged_df.to_csv("abhi_healthcare_project.csv",index=False)

In [130]:
data_file = pd.read_csv("/content/abhi_healthcare_project.csv")
data_file

,beneficiary_id,claim_id,segment,claim_from_date,claim_thru_date,provider_number,claim_payment_amt,primary_payer_claim_paid_amt,blood_deductible_liability_amount,part_b_deductible_amount,...,depression,diabetes,ischemic_heart_disease,osteoporosis,arthritis,stroke,medicare_reimbursement_amt,beneficiary_responsibility_amt,primary_payer_reimbursement_amt,chronic_conditions
0,00001C24EE7B06AC,684562269783396,1,2009-04-04,2009-04-04,1100SK,200.0,0.0,0.0,0.0,...,False,False,True,False,False,False,200.0,40.0,0.0,ischemic_heart_disease
1,00001C24EE7B06AC,684012269893042,1,2010-03-10,2010-03-10,1100SK,500.0,0.0,0.0,0.0,...,False,False,True,False,False,False,200.0,40.0,0.0,ischemic_heart_disease
2,000072CF62193213,684012269540703,1,2008-01-30,2008-01-30,1000AH,50.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,NaN
3,000072CF62193213,684472269696971,1,2008-03-01,2008-03-01,1000AH,70.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,NaN
4,000072CF62193213,684122269778934,1,2008-03-22,2008-03-22,3400HK,40.0,0.0,0.0,0.0,...,False,False,False,False,False,False,130.0,70.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
790039,FFFE6F53545F974E,684022269969505,1,2010-01-09,2010-01-09,3600RB,10.0,0.0,0.0,0.0,...,False,True,True,False,False,False,600.0,50.0,0.0,Multiple
790040,FFFE6F53545F974E,684532270106179,1,2010-02-23,2010-02-23,3600RB,50.0,0.0,0.0,100.0,...,False,True,True,False,False,False,600.0,50.0,0.0,Multiple
790041,FFFE6F53545F974E,684212269796331,1,2010-03-12,2010-03-12,3600RB,100.0,0.0,0.0,0.0,...,False,True,True,False,False,False,600.0,50.0,0.0,Multiple
790042,FFFE6F53545F974E,684722270135712,1,2010-08-05,2010-08-06,3600RB,10.0,0.0,0.0,0.0,...,False,True,True,False,False,False,600.0,50.0,0.0,Multiple
